In [1]:
import numpy as np
import tensorflow as tf 
from tensorflow.keras.preprocessing.text import  Tokenizer
from tensorflow.keras.preprocessing.sequence import  pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input , Embedding , SimpleRNN , Dense

In [2]:
sentences = [
 "I love this product",
 "This movie made me smile",
 "Service was friendly and quick",
 "Today felt bright and happy",
 "This is the best day",
 "Absolutely fantastic experience",
 "I enjoyed every single moment",
 "Great job, well done",
 "The food tasted delicious",
 "Totally recommend to everyone",
 "Very satisfied with results",
 "This worked better than expected",
 "Amazing quality and value",
 "Such a pleasant surprise",
 "I feel positive about this",
 "I hate this product",
 "This movie bored me",
 "Service was rude and slow",
 "Today was cold and lonely",
 "This is the worst day",
 "Terrible experience overall",
 "I regret buying this",
 "Very disappointed with results",
 "The food tasted awful",
 "Do not recommend this",
 "It broke after one use",
 "Not worth the money",
 "Utterly frustrating and annoying",
 "I feel negative about this",
 "Such a waste of time",
]
labels = [1]*15 +[0]*15
labels = np.array(labels)

labels = [1]*15 + [0]*15 — We create the target labels

1 = Positive sentiment (first 15)

0 = Negative sentiment (last 15)

[1]*15 creates [1, 1, 1, ..., 1] (15 ones)

- > (+) concatenates both lists → 30 labels total


labels = np.array(labels) — Converts the Python list to a NumPy array, because Keras/TensorFlow needs NumPy arrays for training, not plain Python lists


This cell creates a small sentiment analysis dataset — 30 sentences with their labels (1=positive, 0=negative) — ready for preprocessing

Tokenization & Padding (Text → Numbers):

In [3]:
vocab_size =2000
tok= Tokenizer(num_words=vocab_size,oov_token= "<00V>")
tok.fit_on_texts(sentences)
seqs=tok.texts_to_sequences(sentences)
maxlen=max(len(s) for s in seqs)

X= pad_sequences(seqs,maxlen=maxlen,padding='post')
y=labels

In [ ]:
1. vocab_size = 2000

-Sets the maximum number of unique words the tokenizer will keep

-If our text had 3000 unique words, only the top 2000 most frequent would be kept, rest would be treated as unknown

-Our dataset is tiny (~60 unique words), so 2000 is more than enough

2. tok = Tokenizer(num_words=vocab_size, oov_token="")

-Creates a Tokenizer object

-num_words=2000 → only keep top 2000 words

-oov_token="" → any word NOT in vocabulary gets replaced with this special token (OOV = Out Of Vocabulary). It gets assigned index 1

3. tok.fit_on_texts(sentences)

-Scans ALL 30 sentences and builds a word → number dictionary (word index)

-Example: {"": 1, "this": 2, "i": 3, "and": 4, ...}

-Most frequent words get the lowest numbers

4. seqs = tok.texts_to_sequences(sentences)

-Converts each sentence from words → list of numbers using the dictionary above

-Example: "I love this product" → [3, 12, 2, 8]

-Now the RNN can read it!
5. maxlen = max(len(s) for s in seqs)

-Finds the longest sentence (in terms of word count)

-In our dataset, some sentences have 5 words, some have 6 → maxlen = length of the longest one

-We need this because RNN needs all inputs to be the same length in a batch

6. X = pad_sequences(seqs, maxlen=maxlen, padding='post')

-Makes ALL sequences the same length by adding 0s at the end

-padding='post' → add zeros after the sentence (at the end)

-Example:

    "I love this"     → [3, 12, 2]       → [3, 12, 2, 0, 0, 0]
    "Great job well done" → [15, 20, 18, 9] → [15, 20, 18, 9, 0, 0]

7. y = labels

Just assigns the labels array to y for cleaner naming — X = inputs, y = targets

In short:

This cell converts raw text sentences into fixed-length number sequences that the RNN can process. Words → Numbers (tokenize) → Same length (pad).

In [4]:
X[0]

array([ 3, 26,  2,  7,  0])

In [5]:
embed_dim = 16
rnn_units = 8

embed_dim = 16 — Word ki "Identity Card"

Right now after tokenization, each word is just one number:

love = 12

hate = 19

good = 5

But a single number can't tell the RNN what the word means. Is "love" close to "like"? Is "hate" close to "bad"? A single number can't capture this.

Solution → Embedding: Convert each word from 1 number into 16 numbers (a vector).

Think of it like an identity card for each word:

Before Embedding (just a roll number):
  love = 12          ← just a number, no meaning

After Embedding (full identity card with 16 traits):

  love = [0.8, 0.9, -0.1, 0.3, 0.7, ...]   ← 16 numbers describing the word

  like = [0.7, 0.85, -0.05, 0.25, 0.6, ...] ← similar to "love"! 

  hate = [-0.8, -0.9, 0.5, -0.3, -0.6, ...] ← very different from "love"


embed_dim = 16 means each word gets 16 traits in its identity card

These traits are learned automatically during training

Why 16? Small dataset → small embedding is enough. For big models like GPT, this is 768 or more!


rnn_units = 8 — RNN ki "Brain Size"

This is how many neurons are in the RNN layer = how much the RNN can remember.

Think of it like brain capacity:

rnn_units = 8  → Small brain (8 memory slots)  → Good for simple tasks

rnn_units = 64 → Medium brain                   → Good for most tasks  

rnn_units = 256 → Big brain                     → Good for complex tasks

At each time step, the hidden state h_t will be a vector of 8 numbers:

h_t = [0.2    ,  -0.5   ,  0.8,  0.1,  -0.3,  0.6,  -0.7,  0.4]

        ↑    ↑    ↑    ↑    ↑     ↑    ↑     ↑

        8 memory slots storing info about all words seen so far

We use 8 because our dataset is tiny (30 sentences) — a big brain on tiny data = overfitting.

In [ ]:
# "I love this" 
#      ↓ (Tokenizer)
#   [3, 12, 2, 0, 0]          ← each word = 1 number
#      ↓ (Embedding, embed_dim=16)
#   [[0.1, 0.3, ...],         ← each word = 16 numbers (identity card)
#    [0.8, 0.9, ...],
#    [0.5, 0.2, ...],
#    [0.0, 0.0, ...],
#    [0.0, 0.0, ...]]
#      ↓ (RNN, rnn_units=8)
#   [0.2, -0.5, 0.8, ...]     ← 8 numbers = RNN's memory after reading all words
#      ↓ (Dense layer)
#   0.92                       ← final prediction (positive!)


In [6]:
inp=Input(shape=(maxlen,),dtype="int32",name='input')
x=Embedding(input_dim=vocab_size,output_dim=embed_dim,mask_zero=True,name='embed')(inp)

rnn = SimpleRNN(units=rnn_units, return_sequences=False,return_state=False,name='simple_rnn')

x_last =rnn(x)

out=Dense(1,activation='sigmoid',name='out')(x_last)

model=Model(inputs=inp,outputs=out)

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
model.summary()



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 5)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 5, 16)     │     32,000 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 5)         │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 8)         │        200 │ embed[0][0],      │
│ (SimpleRNN)         │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ out (Dense)         │ (None, 1)         │          9 │ simple_rnn[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,209 (125.82 KB)

 Trainable params: 32,209 (125.82 KB)

 Non-trainable params: 0 (0.00 B)

# Building the SimpleRNN Model


Let me go line by line:

Let me go line by line:

1. inp = Input(shape=(maxlen,), dtype="int32", name='input')

-  Creates the entry point of the model

-   shape=(maxlen,) → each input is a sequence of maxlen numbers (our padded word indices)

-   dtype="int32" → word indices are whole numbers (3, 12, 2...), not decimals

-   name='input' → just a label for the layer (shows in model.summary)

Example input: [3, 12, 2, 8, 0, 0]  ← one padded sentence


2. x = Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=True, 

name='embed')(inp)

- Converts each word number → 16-number identity card (as we discussed)

- input_dim=vocab_size(2000) → total possible words

- output_dim=embed_dim(16) → each word becomes 16 numbers

- mask_zero=True → tells the model to IGNORE the padded zeros! Very important — without 

this, the RNN would think 0 is a real word and try to learn from it

- (inp) at the end → connects this layer to the input layer

[3, 12, 2, 0, 0] → [[0.1,0.3,...], [0.8,0.9,...], [0.5,0.2,...], IGNORED, IGNORED]

 ---------------------word1         word2         word3        masked   masked

3. rnn = SimpleRNN(units=rnn_units, return_sequences=False, return_state=False, name='simple_rnn')

- Creates a SimpleRNN layer with 8 neurons (our brain)

- return_sequences=False → give me ONLY the last hidden state h_T (not every step's output). Because for sentiment analysis we only need the final summary after reading all words

- return_state=False → don't return the hidden state separately (just the output is enough)

return_sequences=False:

  word1 → [RNN] → word2 → [RNN] → word3 → [RNN] → h3 ✅ (only this returned)
                                                     ↑
                                              8 numbers summarizing
                                              the entire sentence

4. x_last = rnn(x)

- Passes the embedded words through the RNN
- The RNN reads word by word, updating its memory at each step
- x_last = the final hidden state = RNN's summary of the entire sentence
- Shape: (batch_size, 8) → 8 numbers per sentence

5. out = Dense(1, activation='sigmoid', name='out')(x_last)

- The output layer — takes RNN's 8-number summary → produces 1 number
- activation='sigmoid' → squishes output between 0 and 1
- Close to 1 = Positive sentiment
- Close to 0 = Negative sentiment
- x_last = [0.2, -0.5, 0.8, 0.1, -0.3, 0.6, -0.7, 0.4]  (8 numbers)
                              ↓ Dense layer
                           out = 0.92  → Positive! ✅

6. model = Model(inputs=inp, outputs=out)

- Connects everything into one model: input → embedding → RNN → dense → output
- Using Functional API (not Sequential) because it gives more flexibility

7. model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

- Prepares the model for training
- optimizer='adam' → Adam optimizer (smart gradient descent, adjusts learning rate automatically)
- loss='binary_crossentropy' → loss function for binary classification (positive vs negative, 2 classes)
- metrics=['accuracy'] → track accuracy during training
8. model.summary()

- Prints the model architecture — layers, output shapes, parameter count






In [7]:

model.fit(X,y,epochs= 25,batch_size = 8,verbose = 1)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.4333 - loss: 0.6989
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5667 - loss: 0.6833
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6667 - loss: 0.6692
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7667 - loss: 0.6559
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8000 - loss: 0.6440
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8333 - loss: 0.6290 
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9000 - loss: 0.6152
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8667 - loss: 0.6009
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.9333 - loss: 0.5841
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.9000 - loss: 0.5660 
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9333 - loss: 0.5475
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9333 - loss: 0.5275

# Task

Create an intermediate Keras model to visualize the outputs of the embedding layer and the hidden states of the SimpleRNN layer from the existing model. Then, use this intermediate model to predict on the input data and visualize the outputs.

# Create intermediate model

## Subtask:

Build a new Keras Model that takes the same input as the original model but outputs the embedding layer and the hidden states of the SimpleRNN layer.

**Reasoning**: Define an intermediate Keras model with the same input as the original model and outputs from the embedding and simple RNN layers.

In [8]:
intermediate_model = Model(inputs=model.inputs, outputs=[model.get_layer('embed').output, model.get_layer('simple_rnn').output])

In [9]:
from tensorflow.keras.layers import SimpleRNN as SRNN
seq_inp = Input(shape=(maxlen,), dtype='int32')
seq_emb = model.get_layer('embed')(seq_inp)  # reuse trained embedding

# Create RNN with return_sequences=True
rnn_seq = SRNN(units=rnn_units, return_sequences=True, name='rnn_seq')

# DO NOT CALL build() manually
seq_hidden = rnn_seq(seq_emb)  # builds automatically

# Copy trained RNN weights
try:
    trained_weights = model.get_layer('simple_rnn').get_weights()
    rnn_seq.set_weights(trained_weights)
    print("Copied RNN weights into sequence-inspection RNN.")
except Exception as e:
    print("Could not copy weights automatically:", e)

inspect_model = Model(inputs=seq_inp, outputs=seq_hidden)

# Inspect
idx = 0
example_seq = X[idx:idx+1]  # shape (1, maxlen)
hidden_seq = inspect_model.predict(example_seq)

print("Sentence:", sentences[idx])
print("Token ids:", example_seq)
print("Hidden states per timestep shape:", hidden_seq.shape)
print("Hidden states (timesteps x units):")
print(np.round(hidden_seq[0], 3))

Copied RNN weights into sequence-inspection RNN.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 452ms/step
Sentence: I love this product
Token ids: [[ 3 26  2  7  0]]
Hidden states per timestep shape: (1, 5, 8)
Hidden states (timesteps x units):
[[ 0.024 -0.008 -0.008 -0.057 -0.044 -0.054 -0.005 -0.029]
 [-0.347 -0.136 -0.021 -0.065 -0.157  0.103 -0.109  0.103]
 [-0.129 -0.194 -0.105  0.299 -0.01   0.247 -0.04  -0.111]
 [-0.275 -0.305 -0.448  0.018 -0.045  0.173 -0.05   0.142]
 [-0.275 -0.305 -0.448  0.018 -0.045  0.173 -0.05   0.142]]
